In [1]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

In [2]:
mesh = unit_square.GenerateMesh(maxh=0.2)
mesh = Mesh(mesh)
print(f"n vertices: {mesh.nv}")
print(f"n elements: {mesh.ne}")
Draw(mesh)

n vertices: 38
n elements: 54


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [3]:
shape = Rectangle(2, 1).Face()
geom = OCCGeometry(shape, dim=2).GenerateMesh(maxh=0.2)

mesh = Mesh(geom)
print(f"n vertices: {mesh.nv}")
print(f"n elements: {mesh.ne}")
Draw(mesh)

n vertices: 78
n elements: 124


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [4]:
box = Rectangle(1,1).Face()
circ1 = Circle((0.3,0.7), 0.1).Face()
circ2 = Circle((0.7,0.7), 0.1).Face()
bar = MoveTo(0.2,0.2).Rectangle(0.6,0.1).Face()
air = box-circ1-circ2-bar

circ1.faces.name = "left"
circ2.faces.name = "right"
air.faces.name = "air"
bar.faces.name = "bar"
air.edges.Min(Y).name ='b'
air.edges.Max(X).name ='r'

shape = Glue([air,circ1,circ2,bar])
# Draw (shape)

geo = OCCGeometry(shape, dim=2)
mesh = Mesh(geo.GenerateMesh(maxh=0.05)).Curve(3)
Draw (mesh);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

In [5]:
cube = Box((0,0,0),(1,1,1))
cyl = Cylinder((0,0.5,0.5),X, r=0.2, h=1)
cube.faces.name = "outer"
cyl.faces.name = "cyl"
shape = cube-cyl

ngmesh = OCCGeometry(shape).GenerateMesh(maxh=0.1)
mesh = Mesh(ngmesh)
mesh.Curve(3)
Draw (mesh);

for l in range(1):
    ngmesh.Refine()
mesh = Mesh(ngmesh)
mesh.Curve(3)
Draw (mesh);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

In [6]:
# Create the Block (Solid)
block = Box((0,0,0), (1,1,1))
# Create the Tube (Fluid)
tube = Cylinder((0,0.5,0.5), X, r=0.2, h=1)
fluid = Cylinder((0,0.5,0.5), X, r=0.17, h=1)

# IMPORTANT: Cut the tube out of the block first, then glue them
# This creates two "Volumes" that share one "Surface"
solid_part = block - tube
tube_part = tube - fluid
fluid_part = fluid

# Identify them for the solver
solid_part.mat("solid")
tube_part.mat("tube")
fluid_part.mat("fluid")

# Combine into one geometry
fsi_geo = Glue([solid_part, tube_part, fluid_part])

# Generate Mesh
mesh = Mesh(OCCGeometry(fsi_geo).GenerateMesh(maxh=0.1)).Curve(3)

Draw(mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [7]:
def GenerateMeshOld(order, maxh=0.2):
    circle = Circle((0.2, 0.2), r=0.05).Face()
    circle.edges.name = "circ"
    fluid = Rectangle(2.5, 0.41).Face()
    fluid.faces.name = "fluid"
    fluid.edges.Min(X).name = "inlet"
    fluid.edges.Max(X).name = "outlet"
    fluid.edges.Min(Y).name = "wall"
    fluid.edges.Max(Y).name = "wall"
    solid = (
        MoveTo(0.248989794855664, 0.19).Rectangle(0.6 - 0.248989794855664, 0.02).Face()
    )
    solid.faces.name = "solid"
    solid.edges.name = "interface"
    solid.edges.Min(X).name = "circ_inner"

    domain_fluid = (fluid - circle) - solid
    domain = Glue([domain_fluid, solid])

    mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=maxh))
    mesh.Curve(order)

    return mesh

def GenerateMesh(order, maxh=1):
    
    abdomen = Rectangle(30, 15).Face()
    abdomen.faces.name = "abdomen"
    abdomen.edges.Min(X).name = "left"
    abdomen.edges.Max(X).name = "right"
    abdomen.edges.Min(Y).name = "bottom"
    abdomen.edges.Max(Y).name = "top"

    aorta = MoveTo(0, 6.5).Rectangle(30, 3).Face()
    aorta.faces.name = "aorta"
    aorta.edges.Min(X).name = "left"
    aorta.edges.Max(X).name = "right"
    aorta.edges.Min(Y).name = "bottom"
    aorta.edges.Max(Y).name = "top"

    blood = MoveTo(0, 7).Rectangle(30, 2).Face()
    blood.faces.name = "blood"
    blood.edges.Min(X).name = "inlet"
    blood.edges.Max(X).name = "outlet"
    blood.edges.Min(Y).name = "wall"
    blood.edges.Max(Y).name = "wall"
    

    abdomen_domain = abdomen - aorta
    aorta_domain = aorta - blood
    blood_domain = blood
    domain = Glue([abdomen_domain, aorta_domain, blood_domain])
    mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=maxh))
    mesh.Curve(order)

    return mesh


mesh = GenerateMesh(order=3)
Draw(mesh);


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

In [18]:
outer = Rectangle(2, 2).Face()
outer.edges.name="outer"
outer.edges.Max(X).name = "r"
outer.edges.Min(X).name = "l"
outer.edges.Min(Y).name = "b"
outer.edges.Max(Y).name = "t"

inner = MoveTo(1, 0.9).Rectangle(0.3, 0.5).Face()
inner.edges.name="interface"
outer = outer - inner

inner.faces.name="inner"
inner.faces.col = (1, 0, 0)
outer.faces.name="outer"
outer.faces.col = (0.2, 0, 0.5)

geo = Glue([inner, outer])
Draw(geo);

mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=0.2))
Draw(mesh);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

In [29]:
box = Box(p1=(0, 0, 0), p2=(1, 1, 2))
box.mat("box")
box.faces.name = "box"
box.faces.Min(X).name = "left"
box.faces.Max(X).name = "right"
box.faces.Min(Y).name = "bottom"
box.faces.Max(Y).name = "top"
box.faces.Min(Z).name = "rear"
box.faces.Max(Z).name = "front"

cyl = Cylinder(p=(0.5, 0.5, 0), d=Z, r=0.2, h=2)
cyl.mat("tube")
cyl.faces.name = "interface"
cyl.faces.Min(Z).name = "inlet"
cyl.faces.Max(Z).name = "outlet"


shape = box - cyl

ngmesh = OCCGeometry(shape).GenerateMesh(maxh=0.1)
mesh = Mesh(ngmesh)
mesh.Curve(3)
Draw(mesh);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…